In [ ]:
import pandas as pd

df = pd.read_json("/content/sample_data/News_Category_Dataset_v3.json", lines=True)

In [ ]:
print(df.head())
print(df.columns)
print(df.shape)

                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

In [ ]:


print(df['category'].value_counts())

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [ ]:
print(df.isnull().sum())

link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
dtype: int64


In [ ]:
## Question 1 :   Keep only these categories: TECHNO, ENTERTAINMENT, POLITICS, BUSINESS

df = df[df['category'].isin(['TECH','ENTERTAINMENT','POLITICS', 'BUSINESS'])]



In [ ]:
## Question 2 : Load spaCy’s English model without unnecessary components.
import spacy

nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner"])
print(nlp.pipe_names)

['tok2vec', 'attribute_ruler', 'lemmatizer']


In [ ]:
## Question 3: Write a function to preprocess headlines: lowercase, remove stopwords,punctuation, and lemmatize

def preprocess(text):
  a = nlp(text.lower())
  words = []

  for token in a:
    if not token.is_stop and not token.is_punct:
      words.append(token.lemma_)

  return " ".join(words)

print(preprocess("An Apple a day keeps a Doctor away!"))

apple day keeps doctor away


/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [ ]:
# Question 4: Convert text into numeric vectors using CountVectorizer with unigrams and bigrams

from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(ngram_range=(1,2))
X = vec.fit_transform(df['headline'])
print(vec.get_feature_names_out())

['00' '00 00' '000' ... 'zynga ceo' 'zynga stock' 'ᵒᴥᵒᶅ']


In [ ]:
## Question 5: Limit the vocabulary size to 5000 features; explain why

vec = CountVectorizer(max_features=5000)
X = vec.fit_transform(df['headline'])
print(vec.get_feature_names_out())

## We limit the vocabulary to 5000 features to save memory, remove less important words and improve model performance by reducing overfitting

In [ ]:
## Question 6: Create feature matrix X and label vector y.

X = vec.fit_transform(df['headline'])
y = df['category']

In [ ]:
## Question 7: Split data into training and test sets with balanced categories.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
## Question 8: Train Logistic Regression model with enough iterations.

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
## Question 9: Test the model and report accuracy.

from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.8896986570586308


In [ ]:
## Question 10: Build a function to predict categories for new headlines

def predict_category(headline):
  headline = preprocess(headline)
  headline = vec.transform([headline])
  return model.predict(headline)[0]
print(predict_category("Apple releases a new iphone"))

TECH


In [ ]:
## Question 11: Create a Chabot that takes user input and predicts the category until user types 'quit' or 'exit’